# Anthropic API

**Module:** 08-llm-apis

**Notebook:** `05-anthropic-api.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Messages API** with clear contracts and failure modes
- Explain and apply **Content Blocks** with clear contracts and failure modes
- Explain and apply **Tool Use** with clear contracts and failure modes
- Explain and apply **Streaming Events** with clear contracts and failure modes
- Explain and apply **Prompt Caching** with clear contracts and failure modes
- Explain and apply **Usage** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Anthropic API

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Messages API**
2. **Content Blocks**
3. **Tool Use**
4. **Streaming Events**
5. **Prompt Caching**
6. **Usage**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Messages API

### Definition
**Messages API** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Messages API typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Messages API: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Messages API as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Messages API as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Messages API
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Messages API when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Messages API improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Messages API" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Messages API"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


## Content Blocks

### Definition
**Content Blocks** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Content Blocks typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Content Blocks: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Content Blocks as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Content Blocks as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Content Blocks
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Content Blocks when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Content Blocks" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Content Blocks"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Content Blocks"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Content Blocks"}
strong = {"definition": "Content Blocks", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Content Blocks"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Content Blocks", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Content Blocks

**Situation:** A team wants to productionize a feature involving **Content Blocks**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Tool Use

### Definition
**Tool Use** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Tool Use typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tool Use: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Tool Use as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tool Use as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tool Use
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Tool Use when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tool Use" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tool Use"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Streaming Events

### Definition
**Streaming Events** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Streaming Events typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Streaming Events: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Streaming Events as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Streaming Events as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Streaming Events
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Streaming Events when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Streaming Events" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Streaming Events"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def usage_account(prompt: str, completion: str, price_in=0.15, price_out=0.60):
    # prices are illustrative $/1M tokens
    tin, tout = approx_tokens(prompt), approx_tokens(completion)
    cost = (tin * price_in + tout * price_out) / 1_000_000
    return {"prompt_tokens": tin, "completion_tokens": tout, "usd_estimate": round(cost, 6)}

print(usage_account("system+user..." * 50, "answer..." * 20))


In [ ]:
# Streaming chunk assembler (shape similar to provider events)
chunks = [{"delta": "Hello"}, {"delta": ", "}, {"delta": "world"}]
out = []
for ch in chunks:
    out.append(ch["delta"])
    print("partial:", "".join(out))
print("final:", "".join(out))


### Worked scenario — Streaming Events

**Situation:** A team wants to productionize a feature involving **Streaming Events**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Prompt Caching

### Definition
**Prompt Caching** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Prompt Caching typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Prompt Caching: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Prompt Caching as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Prompt Caching as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Prompt Caching
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Prompt Caching when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Prompt Caching" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Prompt Caching"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


## Usage

### Definition
**Usage** is a core building block in 05-anthropic-api within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Usage typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Usage: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Usage as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Usage as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Usage
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Usage when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Usage" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Usage"
    notebook: str = "05-anthropic-api"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def usage_account(prompt: str, completion: str, price_in=0.15, price_out=0.60):
    # prices are illustrative $/1M tokens
    tin, tout = approx_tokens(prompt), approx_tokens(completion)
    cost = (tin * price_in + tout * price_out) / 1_000_000
    return {"prompt_tokens": tin, "completion_tokens": tout, "usd_estimate": round(cost, 6)}

print(usage_account("system+user..." * 50, "answer..." * 20))


In [ ]:
# Streaming chunk assembler (shape similar to provider events)
chunks = [{"delta": "Hello"}, {"delta": ", "}, {"delta": "world"}]
out = []
for ch in chunks:
    out.append(ch["delta"])
    print("partial:", "".join(out))
print("final:", "".join(out))


### Worked scenario — Usage

**Situation:** A team wants to productionize a feature involving **Usage**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Anthropic API**.

| Topic | Do | Don't |
|-------|----|-------|
| Messages API | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Content Blocks | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tool Use | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Streaming Events | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Prompt Caching | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Usage | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Messages API | Key concept covered in this notebook; see its section for definition and pitfalls |
| Content Blocks | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tool Use | Key concept covered in this notebook; see its section for definition and pitfalls |
| Streaming Events | Key concept covered in this notebook; see its section for definition and pitfalls |
| Prompt Caching | Key concept covered in this notebook; see its section for definition and pitfalls |
| Usage | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Anthropic API** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **08-llm-apis**.


## Try It Yourself

1. Implement a failing test/fixture for **Messages API**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Content Blocks**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Tool Use**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Streaming Events**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Prompt Caching**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
